![](https://www.analyticsvidhya.com/blog/wp-content/uploads/2015/01/scikit-learn-logo.png)

## Introduction
> **Thank you SciKits Learn for providing such dataset, it is a very good learning opportunity for beginners, it contains a  reasonable size of data, a few number of outliers, an excessive number of features and needs to be reduced, it was only missing some null values to test the imputing skills**

#  Importing Libiraries and Data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from sklearn import set_config
%matplotlib inline
plt.figure(figsize=(20,15))
set_config(display='diagram')
sns.set_style('darkgrid')
pd.set_option('display.max_columns',0)
pd.set_option('display.max_rows',100)
warnings.filterwarnings('ignore')

In [ ]:
trainX = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv', header=None)
trainY = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv', header=None)
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv', header=None)
trainX.head()

In [ ]:
trainX.shape, trainY.shape, test.shape

# EDA & Visualizations

## Rename features with proper names

> **Data have no named columns, let's rename it all to be COL_## starting with zero**

In [ ]:
cols = ['COL_' + str(x).zfill(2) for x in range(40)]
trainX.columns = cols
test.columns = cols
trainX.sample(5)

In [ ]:
trainY.columns=['label']
train= pd.concat([trainX, trainY], axis=1)
train.describe(include='all').T

> **Ok, all features have no empty data, and label is binary classification, and all features seems normal distributed, let's see in deep**

In [ ]:
sns.countplot(train.label).set_title('Count of label values', fontsize=15)
plt.show()
print(train.label.value_counts())

> **Label is almost balanced, let's see the distribution of remaining features**

In [ ]:
fig, ax = plt.subplots(nrows=8, ncols=5,sharex=True, sharey=True, figsize=(20,30))
idx=0
for i in train.columns[:-1]:
    sns.kdeplot(data=train, x=i, ax=ax[idx//5][idx%5], fill=True).set_title(f'Distribution of {i}')
    idx+=1

> **I meant to unify the range and density while graphing the KDE to see all data in the same scale cause the differences between features is not big, so, the comparisons between the statistical properties of the features will be easy**
>
> **So, all features are in normal distribution but in a different shape (the mean close to zero, std. ranges from 1 to 5 its ok, almost no skewness except some features "maybe because of some outliers",  at last about kurtosis,  most features in a leptokurtic shape, and some in a platykurtic, while very little looks normal (mesokurtic)**

In [ ]:
fig, ax = plt.subplots(nrows=8, ncols=5, figsize=(20,30))
idx=0
for i in train.columns[:-1]:
    sns.boxplot(data=train, y=i, ax=ax[idx//5][idx%5], palette='Reds', whis=1.5).set_title(f'{i}')
    idx+=1

> **Here in the boxplot I can't unify the range between all features to be able to see each box individually, to see the median position and the whiskers, and to clearly see the outliers.**
>
> **Ok, it looks like there are a lot of outliers. I hope they will be in the same indices between features to not to remove a lot of data.**

# Feature Engineering

## Handling outliers

In [ ]:
def get_column_outliers(df, col_name, whisker):
    q1 = df[col_name].quantile(.25)
    q3 = df[col_name].quantile(.75)
    iqr = q3-q1
    lower_limit  = q1 - (whisker * iqr)
    higher_limit = q3 + (whisker * iqr)
    return df[(df[col_name] < lower_limit) | (df[col_name] > higher_limit)].index

def outliers_report(df, wisker):
    outliers = []
    outliers_count = 0
    print('-'*100,'\n', f' Count of outliers for every feature when whiskers at distance = {wisker} '.center(100,'-'),'\n','-'*100,'\n')
    for i in df.columns:
        col_outliers = list(get_column_outliers(df, i, wisker))
        outliers_count += len(col_outliers)
        if len(col_outliers) > 0:
            outliers += col_outliers
            print(f'{i} contain {str(len(col_outliers)).rjust(2)} outlier --> index {col_outliers}')
    print('-'*50)
    outliers = sorted(list(set(outliers)))
    print('number of outliers from all columns is ', outliers_count)
    print('count of distinct outliers indices = ', len(outliers))

def remove_outliers(df, col_name, wisker):
    idx = get_column_outliers(df, col_name, wisker)
    df.drop(idx, inplace=True)
    print(f' {len(idx)} records droped from {col_name} - indices {list(idx)}')

In [ ]:
outliers_report(train, 1.5)

> **Oh no, at the default level of whiskers (1.5 times the IQR far from the nearest quartile) there is 286 outlier, and its not the same indecies between features, they are in 247 different index, I can't remove the quarter of all data**
>
> **Let's go with whiskers some little far, let's try it at position 2 instead of 1.5**

In [ ]:
outliers_report(train, 2)

> **Ok, that is better, removing only 40 index better that 247 one**

In [ ]:
for i in train.columns:
    remove_outliers(train, i, 2)

In [ ]:
fig, ax = plt.subplots(nrows=8, ncols=5, figsize=(20,30))
idx=0
for i in train.columns[:-1]:
    sns.boxplot(data=train, y=i, ax=ax[idx//5][idx%5], whis=2, palette='Greens')
    idx+=1

> **After removing the most far outliers, boxplots looks more better**
>
> **At COL_13 a new outlier appears, but it is ok, not necessary to remove it**

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1,figsize=(20,18))
sns.heatmap(train.corr(), cmap="RdYlBu", annot=True, fmt=".01f")
plt.show()

> **The Corelation matrix tells us there is no highly important feature affecting Label, but there are some few features that have a small affection on Label**
>
> **Now we need to reduce the features count. 40 features is a lot, especially when applying the testing data (9000) it will take so much time**

# Preprocessing

> **Before reducing the number of features, we need to rescale our data first**
>
> **But since all features are already in a  normal distribution, there is no reason to use StandardScaler. Better use the MinMaxScaler. At least it will make our data on the same scale.**

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler().fit(train)
train_scaled = pd.DataFrame(scaler.transform(train), columns=cols+['label'])

In [ ]:
fig, ax = plt.subplots(nrows=8, ncols=5,sharex=True, sharey=True, figsize=(20,30))
idx=0
for i in train.columns[:-1]:
    sns.kdeplot(data=train_scaled, x=i,hue='label' , ax=ax[idx//5][idx%5], color='green', fill=True ).set_title(f'Distribution of {i}')
    idx+=1

> **Very Good, with unifiying the range and the density between all features, the KDE shape is almost the same between all, thank you MinMaxScaler**

In [ ]:
fig, ax = plt.subplots(nrows=8, ncols=5, figsize=(20,30), sharex=True, sharey=True)
idx=0
for i in train.columns[:-1]:
    sns.boxplot(data=train_scaled,x='label', y=i, ax=ax[idx//5][idx%5], palette='Accent')
    idx+=1

> **while viewing the boxplot after splitting it according to Labels, (even if unifing the range) it looks good**
>
> **But we must not forget, there are still outliers because we removed only the very far ones, not all outliers.**

## Time to import SciKit Learn

> **Now, it is time to get SciKitsLearn our host, and to respect it I'll not import any other module today.**

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.metrics         import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition   import PCA
from sklearn.pipeline        import Pipeline

from sklearn.linear_model    import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes     import GaussianNB
from sklearn.svm             import SVC
from sklearn.ensemble        import RandomForestClassifier
from sklearn.tree            import DecisionTreeClassifier
from sklearn.neighbors       import KNeighborsClassifier
from sklearn.neural_network  import MLPClassifier

## PCA (Choose a better number of features)

In [ ]:
train_newX = train_scaled.iloc[:,:-1]
train_newY = train_scaled.label
pca = PCA(n_components=30).fit(train_newX)
plt.plot(pca.explained_variance_, 'r-o')
plt.title('Scree Plot',fontsize=14)
plt.show()

> **It seems the more we take components, the much we get better explanation.**
>
> **But to take advantage of PCA let's suffice with 20 only, and unfortunately dispense with the better explanation (and for sure it will affect the model accuracy).**

In [ ]:
pca = PCA(n_components=20).fit(train_newX)
train_newX = pca.transform(train_newX)

fig, ax = plt.subplots(nrows=1, ncols=1,figsize=(15,10))
sns.heatmap(pd.concat([pd.DataFrame(train_newX),train_newY],axis=1).corr(), cmap="RdYlBu", annot=True, fmt=".02f")
plt.show()

> **Not big change, very few efficiency features, but the total count of features is now one half the original count**

## Splitting to train and validation groups

In [ ]:
xtrain, xvalid, ytrain, yvalid = train_test_split(train_newX, train_newY, test_size=.3, random_state=30)
xtrain.shape, xvalid.shape, ytrain.shape, yvalid.shape

## Choosing the best algorithm and tune hyperparameters

> **Let's start some algorithms and tune its hyperparameters to find the best of each estimator**

In [ ]:
result_dict = dict()
def check_model(model_name, hypers):
    model = str(model_name).split('(')[0]
    gridmodel = GridSearchCV(estimator = model_name, param_grid = hypers, cv=10, return_train_score=True)
    gridmodel.fit(xtrain, ytrain)
    ypred = gridmodel.predict(xvalid)    
    time = np.mean(gridmodel.cv_results_['mean_fit_time'] + gridmodel.cv_results_['mean_score_time']) * gridmodel.n_splits_ * len(gridmodel.cv_results_['mean_fit_time'])
    print(f'By model --> {model} using parameters {gridmodel.best_params_}')
    print(f'it tooks {time:0.02f} seconds - and get accuracy = {round(gridmodel.best_score_ * 100,2)} %')
    print('='*50)
    plt.figure(figsize=(5,3))
    sns.heatmap(confusion_matrix(yvalid,ypred), annot=True, fmt='d', cmap='Greens',cbar=False).set_title(f'{model} - Confusion Matrix', fontsize=16)
    plt.show()
    result_dict[model] = (gridmodel.best_score_)

In [ ]:
params_dict = {'penalty' : ('l1', 'l2', 'elasticnet'), 
               'tol': (.0001, .001, .01, .1, 1, 10, 20, 50), 
               'solver' : ('newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga'), 
               'multi_class': ('auto', 'ovr', 'multinomial') }

check_model(LogisticRegression(max_iter=500), params_dict)

In [ ]:
params_dict = {'alpha' : [.1,.5, 1, 2],
               'tol': (.0001, .001, .01, .1, 1, 10, 20, 50) ,
               'solver': ('auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs')}

check_model(RidgeClassifier(max_iter=-1), params_dict)

In [ ]:
params_dict = {'kernel': ('linear', 'poly', 'rbf'),
               'tol': (1e-5, 1e-4, 1e-3, 1e-2, 1e-1)}

check_model(SVC(max_iter=-1, gamma='scale'), params_dict)

In [ ]:
params_dict = {'n_neighbors': (5, 10, 15, 20),
               'algorithm' : ('auto', 'ball_tree', 'kd_tree', 'brute')}
check_model(KNeighborsClassifier(p=1), params_dict)

In [ ]:
params_dict = {'var_smoothing': (1e-9, 1e-10, 1e-11)}
check_model(GaussianNB(), params_dict)

In [ ]:
params_dict = {'activation': ('identity', 'logistic', 'tanh', 'relu'),
               'solver' : ('lbfgs', 'sgd', 'adam')}
check_model(MLPClassifier(), params_dict)

In [ ]:
params_dict = {'criterion': ("gini", "entropy"),
               'splitter' : ("best", "random")}
check_model(DecisionTreeClassifier(), params_dict)

In [ ]:
params_dict = {'criterion': ("gini", "entropy"),
               'ccp_alpha' : (.001, .005, .01, .05)}
check_model(RandomForestClassifier(), params_dict)

> **Some algorithms took a long time and unfortunately didn't give us better accuracy**
>
> **I'm a bit not full satisfied with this results maybe because of PCA, and not removing all outliers, but I can't do more better for now, it's ok**

In [ ]:
pd.Series(result_dict).plot(kind='barh', title='Algorithms Scores', colormap ='Accent', figsize=(10,8), fontsize=12)
plt.show()

> **SVC recorded the better accuracy betwen all other algorithms**
>
> **Let's build our model using the chosen parameters for SVC**

In [ ]:
SVC_model = SVC(kernel='rbf', tol=.1).fit(xtrain, ytrain)
print(classification_report(ytrain,SVC_model.predict(xtrain)))

> **Prediction of our model on the same trained data is only 94% (not good), I hope it will be good with the test data**

# Pipeline & Model

![](https://res.cloudinary.com/dltmvmpuz/image/upload/f_auto,w_750/eloquentarduino.com/color-classification/pipeline-example.jpeg)

> **We only used resclaing and PCA, and for sure the final estimator, let's build our pipeline**

In [ ]:
pipe = Pipeline(steps=[('Rescaling', MinMaxScaler()), ('PCA',PCA(n_components=20)), ('SVC',SVC(kernel='rbf', tol=1e-5))])
pipe.fit(train.iloc[:,:-1], train.label)

# Submission

In [ ]:
submit_dict = {'Id': np.arange(1,9001), 'Solution' : pd.Series(pipe.predict(test))}
my_submission = pd.DataFrame(submit_dict)
my_submission.to_csv('submission.csv', index=False)
my_submission

## I'd Love to hear any comment about this notebook, and I'll be thankful if Upvoted, 
## Thank you for supporting